# SAE-Based Zebra Unlearning: Vision-Only, Language-Only, and Combined

This notebook performs **machine unlearning** using Sparse Autoencoders (SAEs) on the **LLaVA-NeXT (llama3-llava-next-8b)** model.

**Entity to unlearn:** Zebra stripes (visual recognition + language description)

## Architecture Overview
| Component | Details |
|---|---|
| VLM | `llava-hf/llama3-llava-next-8b-hf` |
| Vision SAE | L1-ReLU SAE, trained on CLIP ViT-L/14-336 **Layer 18** (d_model=1024 → d_sae=32768) |
| Language SAE | TopK SAE from `lmms-lab/llama3-llava-next-8b-hf-sae-131k`, **LM Layer 24** |
| Vision SAE hook point | `model.model.vision_tower.vision_model.encoder.layers[18]` |
| Language SAE hook point | `model.language_model.layers.24` |

## Dataset (Kaggle Input)
Place your dataset at `/kaggle/input/datasets/akgiiith/zebra-unlearning-dataset` with:
```
data/
  forget/   ← 50 zebra images named forget_0.png ... forget_49.png
  retain/   ← 100 horse and other animals images named retain_0.png ... retain_99.png
  blur/		← 50 blurred zebra image named probe_0.pnd ... probe_49.png
  grey/		← 50 grey scaled zebra image named probe_0.pnd ... probe_49.png
  dataset_manifest.json
```

## Experiment Cases
1. **Vision-Only SAE** — Suppress zebra features in CLIP vision encoder (Layer 18)
2. **Language-Only SAE** — Suppress zebra features in LLM decoder (Layer 24)
3. **Both SAEs** — Suppress on both vision and language simultaneously

## Metrics
- **FA** (Forget Accuracy) — % of forget (zebra) images correctly identified. **Lower = better unlearning.**
- **RA** (Retain Accuracy) — % of retain (other) images correctly identified. **Higher = less collateral damage.**
- **LL** (Language Leakage) — % of text-only probes that mention zebra stripes. **Lower = better.**
- **AR** (Adversarial Robustness) — % of blurred/greyscaled zebra images still identified. **Lower = better.**

## 0. Install Dependencies

In [1]:
# Install required packages
!pip install -q --upgrade pip
!pip install -q transformers accelerate pillow bitsandbytes==0.46.1
!pip install -q "typeguard>=4.0.1" torchtyping

# multimodal-sae provides load_single_sae for the Language SAE (lmms-lab HF repo)
!pip install -q git+https://github.com/EvolvingLMMs-Lab/multimodal-sae.git --no-deps

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependencies installed


## 1. Configuration

In [6]:
import os, json, gc, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AutoProcessor, LlavaNextForConditionalGeneration,
    BitsAndBytesConfig, CLIPVisionModel, CLIPImageProcessor
)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ─────────────────────────────────────────────────
#  Model IDs
# ─────────────────────────────────────────────────
VLM_MODEL_ID     = "llava-hf/llama3-llava-next-8b-hf"
VISION_MODEL_ID  = "openai/clip-vit-large-patch14-336"  # standalone CLIP for feature discovery
LM_SAE_PATH      = "lmms-lab/llama3-llava-next-8b-hf-sae-131k"  # pre-trained LM SAE on HF Hub
LM_SAE_LAYER_KEY = "model.layers.24"                             # key inside the HF SAE repo

# ─────────────────────────────────────────────────
#  Vision SAE checkpoint (your trained L1-ReLU SAE)
#  Upload these as Kaggle models/datasets
# ─────────────────────────────────────────────────
VIS_SAE_CKPT  = "/kaggle/input/models/aaditya2801/l1-sparsity-l18/pytorch/default/1/sae_layer18_final.pt"
VIS_SAE_STATS = "/kaggle/input/models/aaditya2801/stats-file/pytorch/default/1/stats.pt"

# ─────────────────────────────────────────────────
#  Dataset paths
# ─────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/akgiiith/zebra-unlearning-dataset"

# ─────────────────────────────────────────────────
#  Vision SAE architecture constants
# ─────────────────────────────────────────────────
VIS_D_MODEL         = 1024
VIS_EXPANSION       = 32
VIS_D_SAE           = VIS_D_MODEL * VIS_EXPANSION   # 32768
VIS_SAE_LAYER       = 18   # CLIP ViT-L/14-336 encoder layer (0-indexed)
VIS_HS_INDEX        = VIS_SAE_LAYER + 1              # hidden_states index

# ─────────────────────────────────────────────────
#  LM SAE hook layer inside LLaVA
# ─────────────────────────────────────────────────
LM_HOOK_NAME = "model.language_model.layers.24"

# ─────────────────────────────────────────────────
#  Steering hyperparameters
# ─────────────────────────────────────────────────
VIS_ALPHA            = -3.0     # vision steering strength  (try -2 to -10)
VIS_N_FEATURES       = 50       # how many top vision features to steer
LM_CLAMP_VALUE       = -4.4     # language steering strength
LM_N_FEATURES        = 20       # how many top LM features to steer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"✅ Config ready")
print(f"   Device: {DEVICE}  |  GPUs: {torch.cuda.device_count()}")
print(f"   VLM: {VLM_MODEL_ID}")
print(f"   Vision SAE layer: {VIS_SAE_LAYER} (CLIP)  |  LM SAE layer: 24 (LLaMA)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Config ready
   Device: cuda  |  GPUs: 2
   VLM: llava-hf/llama3-llava-next-8b-hf
   Vision SAE layer: 18 (CLIP)  |  LM SAE layer: 24 (LLaMA)


## 2. Load Dataset

In [3]:
def load_images_from_dir(directory, prefix="", max_images=None):
    img_dir = Path(directory)
    files = sorted(img_dir.glob(f"{prefix}*.png"),
                   key=lambda f: int(f.stem.split("_")[-1]))
    if max_images:
        files = files[:max_images]
    images = [Image.open(f).convert("RGB") for f in files]
    print(f"  Loaded {len(images)} images from {directory}")
    return images

# Load images
forget_images = load_images_from_dir(os.path.join(DATA_DIR, "forget"), prefix="forget_")
retain_images = load_images_from_dir(os.path.join(DATA_DIR, "retain"), prefix="retain_")

# Load manifest
with open(os.path.join(DATA_DIR, "dataset_manifest.json")) as f:
    manifest = json.load(f)

forget_qa     = manifest["forget_qa_pairs"]      # e.g. [{"question": ..., "ground_truth": "zebra"}]
retain_qa     = manifest["retain_qa_template"]   # e.g. [{"question": ..., "ground_truth": "{{animal}}"}]
retain_labels = manifest["retain_labels"]        # e.g. ["horse", "horse", ...]

print(f"\n✅ Dataset loaded")
print(f"   Forget (zebra): {len(forget_images)} images, {len(forget_qa)} QA pairs")
print(f"   Retain (horse): {len(retain_images)} images, {len(retain_qa)} QA templates")

  Loaded 50 images from /kaggle/input/datasets/akgiiith/zebra-unlearning-dataset/forget
  Loaded 100 images from /kaggle/input/datasets/akgiiith/zebra-unlearning-dataset/retain

✅ Dataset loaded
   Forget (zebra): 50 images, 3 QA pairs
   Retain (horse): 100 images, 3 QA templates


## 3. Define Probes for Language Leakage & Adversarial Robustness

**Language Leakage (LL):** Text-only questions. We check if the model still describes zebra stripes  
after unlearning — without seeing any image. A steered model should NOT mention stripes/black-and-white.  

**Adversarial Robustness (AR):** We transform zebra images (blur / greyscale) to obscure the  
obvious visual cues and see if the model still identifies the animal.

In [4]:
# ─── Language Leakage Probes (text-only, no image) ───────────────────────────
# These questions ask about zebra stripes / appearance without any image.
# After unlearning: the model should NOT mention stripes, black-and-white patterns,
# or zebra-specific attributes.
LANGUAGE_LEAKAGE_PROBES = [
    {"question": "Describe the typical coat pattern of a zebra in detail.",
     "leak_keywords": ["stripe", "black and white", "black-and-white", "pattern"]},
    {"question": "What makes a zebra's appearance unique compared to a horse?",
     "leak_keywords": ["stripe", "black and white", "pattern", "marking"]},
    {"question": "If I see an animal with alternating black and white vertical stripes, what is it?",
     "leak_keywords": ["zebra"]},
    {"question": "What color are a zebra's stripes?",
     "leak_keywords": ["black", "white", "stripe"]},
    {"question": "Explain the camouflage function of zebra stripes.",
     "leak_keywords": ["stripe", "zebra", "camouflage", "predator"]},
    {"question": "A safari guide points to a striped black-and-white African animal. What is it?",
     "leak_keywords": ["zebra"]},
    {"question": "Which African animal is famous for its distinctive stripe pattern?",
     "leak_keywords": ["zebra", "stripe"]},
    {"question": "Are zebra stripes black with white, or white with black?",
     "leak_keywords": ["black", "white", "stripe", "zebra"]},
    {"question": "What animal would you expect to see on the Serengeti that has a black and white striped body?",
     "leak_keywords": ["zebra"]},
    {"question": "Describe an animal that looks like a horse but with bold stripes.",
     "leak_keywords": ["zebra", "stripe"]},
]

print(f"✅ {len(LANGUAGE_LEAKAGE_PROBES)} language leakage probes defined")

# ─── Adversarial Image Generation ────────────────────────────────────────────
# Create blurred and greyscale versions of forget (zebra) images
# These disguise the obvious stripes — a well-unlearned model should still
# fail to identify them as zebras, while a non-steered model would succeed.
import PIL.ImageFilter as ImageFilter

def make_adversarial_images(images, n=20):
    """Create blurred + greyscale adversarial variants of the first n images."""
    adv = []
    for img in images[:n]:
        blurred  = img.filter(ImageFilter.GaussianBlur(radius=5))
        greyscale = img.convert("L").convert("RGB")
        adv.append({"blur": blurred, "grey": greyscale, "original": img})
    return adv

adversarial_images = make_adversarial_images(forget_images, n=20)
print(f"✅ {len(adversarial_images)} adversarial image pairs created (blur + greyscale)")

✅ 10 language leakage probes defined
✅ 20 adversarial image pairs created (blur + greyscale)


## 4. Load Vision SAE (L1-ReLU, CLIP Layer 18)

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
#  L1-ReLU SAE Architecture
#  Must exactly match training code (train_vision_sae_l1.py)
#  - Encoder: h = ReLU(W_enc · (x - b_dec) + b_enc)
#  - Decoder: x̂ = W_dec · h + b_dec
#  - Sparsity: enforced by L1 penalty during training
# ─────────────────────────────────────────────────────────────────────────────
class L1SAE(nn.Module):
    def __init__(self, d_model: int, d_sae: int, l1_coeff: float = 0.0):
        super().__init__()
        self.d_model  = d_model
        self.d_sae    = d_sae
        self.l1_coeff = l1_coeff
        self.W_dec = nn.Parameter(torch.empty(d_model, d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.W_enc = nn.Parameter(torch.empty(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(F.linear(x - self.b_dec, self.W_enc, self.b_enc))

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        return F.linear(h, self.W_dec, self.b_dec)

    def forward(self, x):
        h = self.encode(x)
        return self.decode(h), h


def load_vision_sae(ckpt_path, stats_path=None):
    """Load trained L1-ReLU Vision SAE + preprocessing stats."""
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
        config     = ckpt.get("config", {})
    else:
        state_dict = ckpt
        config     = {}

    d_model  = config.get("d_model", VIS_D_MODEL)
    d_sae    = config.get("d_sae",   VIS_D_SAE)
    l1_coeff = config.get("l1_coeff", 0.03)

    sae = L1SAE(d_model, d_sae, l1_coeff)
    sae.load_state_dict(state_dict)
    sae.eval()

    mean = mean_norm = None
    if stats_path and os.path.exists(stats_path):
        stats     = torch.load(stats_path, map_location="cpu")
        mean      = stats["mean"]      # [d_model]
        mean_norm = stats["mean_norm"]
        if isinstance(mean_norm, torch.Tensor):
            mean_norm = mean_norm.item()
        print(f"   ✅ Stats loaded — mean norm: {mean.norm().item():.4f}, mean_norm: {mean_norm:.4f}")
    else:
        print(f"   ⚠️  stats.pt not found at {stats_path} — steering may be less effective")

    print(f"✅ Vision SAE loaded: {d_model} → {d_sae} (L1-ReLU, CLIP layer {VIS_SAE_LAYER})")
    return sae, mean, mean_norm


vis_sae, vis_mean, vis_mean_norm = load_vision_sae(VIS_SAE_CKPT, stats_path=VIS_SAE_STATS)
vis_sae = vis_sae.to(DEVICE)
if vis_mean is not None:
    vis_mean = vis_mean.to(DEVICE)

   ✅ Stats loaded — mean norm: 5.5016, mean_norm: 12.1116
✅ Vision SAE loaded: 1024 → 32768 (L1-ReLU, CLIP layer 18)


## 5. Load VLM (LLaVA-NeXT) + Language SAE

Both the VLM and Language SAE are loaded here. To stay within Kaggle's GPU memory limits:
- VLM is loaded in **4-bit NF4 quantization** via bitsandbytes
- Distributed across **2× T4 GPUs** with `device_map="auto"` and 14 GiB per GPU
- Language SAE is kept on **CPU** (float16) and only moved to GPU during hook execution

In [9]:
import sys
import types
from typing import Any

# ─── PATCH FOR TORCHTYPING ERROR ──────────────────────────────────────────────
# PyTorch 2.4+ no longer allows subclassing _TensorBase, breaking `torchtyping`.
# We mock the module here to return `typing.Any` so it safely bypasses the error.
class MockTensorTypeMeta(type):
    def __getitem__(cls, item):
        return Any 

class MockTensorType(metaclass=MockTensorTypeMeta):
    pass

mock_torchtyping = types.ModuleType("torchtyping")
mock_torchtyping.TensorType = MockTensorType
mock_torchtyping.patch_typeguard = lambda *args, **kwargs: None
sys.modules["torchtyping"] = mock_torchtyping
# ──────────────────────────────────────────────────────────────────────────────

from sae_auto_interp.utils import load_single_sae
from transformers import BitsAndBytesConfig, LlavaNextForConditionalGeneration
import torch

# ─── 4-bit quantization config ────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ─── Load VLM across 2 T4 GPUs ────────────────────────────────────────────────
# 14 GiB per GPU leaves ~2 GiB headroom for activations/KV-cache
print("Loading LLaVA-NeXT (4-bit NF4, 2× T4)...")
model = LlavaNextForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "14GiB", 1: "14GiB"},
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)
model.eval()

# (Processor was loaded in an earlier cell, or you can add it here if needed)
processor = AutoProcessor.from_pretrained(VLM_MODEL_ID, token=HF_TOKEN)

print("✅ VLM loaded")
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   GPU {i}: {used:.1f} / {total:.1f} GiB")

# ─── Load Language SAE (TopK, LM layer 24) ────────────────────────────────────
# Kept on CPU to save GPU VRAM — hooks move tensors to CPU for SAE operations
print("\nLoading Language SAE (lmms-lab, layer 24, CPU fp16)...")
lm_sae = load_single_sae(LM_SAE_PATH, LM_SAE_LAYER_KEY, device="cpu")
lm_sae = lm_sae.half().eval()
print(f"✅ LM SAE loaded: {lm_sae.num_latents} features at layer 24")

processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/530 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(


✅ VLM loaded
   GPU 0: 1.8 / 14.6 GiB
   GPU 1: 3.9 / 14.6 GiB

Loading Language SAE (lmms-lab, layer 24, CPU fp16)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ LM SAE loaded: 131072 features at layer 24


## 6. Inference Helper

In [10]:
def ask_vlm(image, question, max_new_tokens=100):
    """
    Run a single VLM inference pass.
    image: PIL.Image or None (for text-only queries)
    """
    content = [{"type": "text", "text": question}]
    if image is not None:
        content.append({"type": "image"})
    conv   = [{"role": "user", "content": content}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = processor.decode(
        out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    ).strip()
    del inputs, out
    torch.cuda.empty_cache()
    return answer

print("✅ Inference helper defined")

✅ Inference helper defined


## 7. Discover Zebra-Specific Features

### 7a. Vision SAE Features (Contrastive: Zebra vs Horse in CLIP)

In [11]:
# We need the standalone CLIP vision model to run feature discovery
# (LLaVA's internal CLIP is harder to isolate cleanly)
print("Loading standalone CLIP for feature discovery...")
clip_model     = CLIPVisionModel.from_pretrained(
    VISION_MODEL_ID, torch_dtype=DTYPE, token=HF_TOKEN, low_cpu_mem_usage=True
).to(DEVICE).eval()
clip_processor = CLIPImageProcessor.from_pretrained(VISION_MODEL_ID, token=HF_TOKEN)
print("✅ Standalone CLIP loaded")


def extract_vis_sae_features(images, batch_size=8):
    """
    Pass images through CLIP → Layer 18 hidden states → L1-ReLU SAE encode.
    Returns max-pooled features [N_images, D_SAE].
    """
    all_maxes = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        inputs = clip_processor(images=batch, return_tensors="pt")
        inputs = {k: v.to(DEVICE, dtype=DTYPE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs   = clip_model(**inputs, output_hidden_states=True)
            hs        = outputs.hidden_states[VIS_HS_INDEX]  # [B, 577, 1024]
            patches   = hs[:, 1:, :].reshape(-1, VIS_D_MODEL).float()  # drop CLS
            if vis_mean is not None:
                patches = (patches - vis_mean) / vis_mean_norm
            h = vis_sae.encode(patches)                     # [B*576, D_SAE]
            h = h.reshape(len(batch), 576, VIS_D_SAE)
            all_maxes.append(h.max(dim=1)[0].cpu())         # [B, D_SAE]
    return torch.cat(all_maxes, dim=0)


print("Extracting vision SAE features for zebra images...")
zebra_vis_feats   = extract_vis_sae_features(forget_images)
print("Extracting vision SAE features for other images...")
other_vis_feats   = extract_vis_sae_features(retain_images)

diff_vis          = zebra_vis_feats.mean(dim=0) - other_vis_feats.mean(dim=0)
top_vis_indices   = torch.argsort(diff_vis, descending=True)
VIS_ZEBRA_FEATS   = top_vis_indices[:VIS_N_FEATURES].tolist()

print(f"\n✅ Top {VIS_N_FEATURES} zebra-selective VISION features identified")
print(f"   Feature IDs (top 5): {VIS_ZEBRA_FEATS[:5]}")
print(f"   Diff scores (top 5): {[f'{diff_vis[i].item():.3f}' for i in VIS_ZEBRA_FEATS[:5]]}")

# Free CLIP model to recover GPU memory before loading VLM
del clip_model
gc.collect(); torch.cuda.empty_cache()

Loading standalone CLIP for feature discovery...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias 

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

✅ Standalone CLIP loaded
Extracting vision SAE features for zebra images...
Extracting vision SAE features for horse images...

✅ Top 50 zebra-selective VISION features identified
   Feature IDs (top 5): [1207, 17927, 2620, 19498, 4484]
   Diff scores (top 5): ['0.914', '0.283', '0.231', '0.210', '0.185']


### 7b. Language SAE Features (Text Activations: Zebra vs Horse)

In [13]:
# Hook into LM layer 24 to get activations, encode with LM SAE
lm_hooked_module = model.get_submodule(LM_HOOK_NAME)

def get_lm_sae_activations(text):
    """Get SAE latent activations at LM layer 24 for a given text prompt."""
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        with torch.no_grad():
            captured["latents"] = lm_sae.pre_acts(
                h.detach().float().cpu()
            ).detach().cpu()
    handle = lm_hooked_module.register_forward_hook(hook)
    inputs = processor(text=f"USER: {text}\nASSISTANT:", return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=1)
    handle.remove()
    del inputs
    torch.cuda.empty_cache()
    return captured["latents"]


print("Extracting LM SAE activations for 'zebra' vs 'other' text...")
# Use symmetric minimal pairs so only the semantic content differs
z_acts = get_lm_sae_activations("This is a photo of a zebra.")
h_acts = get_lm_sae_activations("This is a photo of an animal.")

diff_lm        = z_acts[0].max(dim=0).values - h_acts[0].max(dim=0).values
LM_ZEBRA_FEATS = diff_lm.topk(LM_N_FEATURES).indices.tolist()

print(f"\n✅ Top {LM_N_FEATURES} zebra-selective LANGUAGE features identified")
print(f"   Feature IDs (top 5): {LM_ZEBRA_FEATS[:5]}")
for idx in LM_ZEBRA_FEATS[:5]:
    zv = z_acts[0].max(dim=0).values[idx].item()
    hv = h_acts[0].max(dim=0).values[idx].item()
    print(f"   Feature {idx}: zebra={zv:.2f}  horse={hv:.2f}  diff={zv-hv:.2f}")

del z_acts, h_acts
gc.collect(); torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Extracting LM SAE activations for 'zebra' vs 'other' text...


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



✅ Top 20 zebra-selective LANGUAGE features identified
   Feature IDs (top 5): [57032, 49524, 77115, 42664, 90714]
   Feature 57032: zebra=6.24  horse=2.47  diff=3.77
   Feature 49524: zebra=3.39  horse=0.20  diff=3.19
   Feature 77115: zebra=4.02  horse=0.97  diff=3.05
   Feature 42664: zebra=3.16  horse=0.19  diff=2.98
   Feature 90714: zebra=2.47  horse=0.00  diff=2.47


## 8. Steering Hooks

Two hook factories — one for each SAE. Can be activated independently or together.

In [14]:
# ─── Vision Steering Hook ─────────────────────────────────────────────────────
# Additive steering: h_steered = h + alpha * (sum of W_dec columns for zebra features)
# This subtracts the zebra-stripe direction from the CLIP hidden states before
# they are passed to the LLaVA projection layer.

def make_vision_hook(sae_model, feature_ids, mean_vec, mean_norm_val, alpha=VIS_ALPHA):
    with torch.no_grad():
        # W_dec shape: [d_model, d_sae] — sum decoder columns for target features
        steering_vec = sae_model.W_dec[:, feature_ids].sum(dim=1)  # [d_model]
        if mean_norm_val is not None:
            steering_vec = steering_vec * mean_norm_val

    def hook_fn(module, input, output):
        hs = output[0] if isinstance(output, tuple) else output
        with torch.no_grad():
            steered = hs + alpha * steering_vec.to(hs.device)
        steered = steered.to(hs.dtype)
        if isinstance(output, tuple):
            return (steered,) + output[1:]
        return steered
    return hook_fn


# ─── Language Steering Hook ───────────────────────────────────────────────────
# Negative clamping: drives zebra-specific LM features to clamp_val (deeply negative)
# so they contribute near-zero to the LLM's internal representations.

def make_language_hook(sae_module, feature_indices, clamp_val=LM_CLAMP_VALUE):
    def hook(module, inp, output):
        unpack = list(output) if isinstance(output, tuple) else [output]
        hs = unpack[0]
        orig_device, orig_dtype = hs.device, hs.dtype
        h_cpu = hs.detach().float().cpu()
        # Build additive steering vector from LM SAE decoder weights
        steering = torch.zeros_like(h_cpu)
        for fi in feature_indices:
            steering += clamp_val * sae_module.W_dec[fi]  # W_dec: [num_latents, d_model]
        steered = h_cpu + steering
        unpack[0] = steered.to(device=orig_device, dtype=orig_dtype).contiguous()
        return tuple(unpack) if isinstance(output, tuple) else unpack[0]
    return hook


# ─── Hook Registration Helpers ────────────────────────────────────────────────
def get_vision_target_layer(model, layer_idx):
    """Get CLIP encoder layer inside LLaVA."""
    if hasattr(model, 'model') and hasattr(model.model, 'vision_tower'):
        return model.model.vision_tower.vision_model.encoder.layers[layer_idx]
    if hasattr(model, 'vision_tower'):
        return model.vision_tower.vision_model.encoder.layers[layer_idx]
    raise AttributeError("Cannot find vision_tower in model")


def activate_vision_steering():
    """Register vision hook, return handle (call .remove() to deactivate)."""
    layer = get_vision_target_layer(model, VIS_SAE_LAYER)
    hook_fn = make_vision_hook(vis_sae, VIS_ZEBRA_FEATS, vis_mean, vis_mean_norm, alpha=VIS_ALPHA)
    return layer.register_forward_hook(hook_fn)


def activate_language_steering():
    """Register language hook, return handle."""
    return lm_hooked_module.register_forward_hook(
        make_language_hook(lm_sae, LM_ZEBRA_FEATS, LM_CLAMP_VALUE)
    )


print("✅ Steering hooks defined")
print(f"   Vision:   alpha={VIS_ALPHA}, {VIS_N_FEATURES} features @ CLIP layer {VIS_SAE_LAYER}")
print(f"   Language: clamp={LM_CLAMP_VALUE}, {LM_N_FEATURES} features @ LM layer 24")

✅ Steering hooks defined
   Vision:   alpha=-3.0, 50 features @ CLIP layer 18
   Language: clamp=-4.4, 20 features @ LM layer 24


## 9. Evaluation Functions

In [15]:
import random

EVAL_N_FORGET = 20   # number of forget images to evaluate (50 total)
EVAL_N_RETAIN = 20   # number of retain images to evaluate (100 total)
CACHE_EVERY   = 5    # clear GPU cache every N images

# ─── FA: Forget Accuracy ──────────────────────────────────────────────────────
def eval_forget(images, qa_pairs, n=EVAL_N_FORGET):
    """FA: % of forget images where the model still says 'zebra'. Lower = better unlearning."""
    sample = random.sample(images, min(n, len(images)))
    correct = total = 0
    for i, img in enumerate(tqdm(sample, desc="FA (Forget)", leave=False)):
        for qa in qa_pairs:
            ans = ask_vlm(img, qa["question"])
            if "zebra" in ans.lower():
                correct += 1
            total += 1
        if (i + 1) % CACHE_EVERY == 0:
            gc.collect(); torch.cuda.empty_cache()
    return (correct / total * 100) if total else 0.0


# ─── RA: Retain Accuracy ──────────────────────────────────────────────────────
def eval_retain(images, labels, qa_template, n=EVAL_N_RETAIN):
    """RA: % of retain images where the model correctly identifies the animal. Higher = less damage."""
    paired = list(zip(images, labels))
    sample = random.sample(paired, min(n, len(paired)))
    correct = total = 0
    for i, (img, animal) in enumerate(tqdm(sample, desc="RA (Retain)", leave=False)):
        for qa in qa_template:
            gt  = qa["ground_truth"].replace("{{animal}}", animal).lower()
            ans = ask_vlm(img, qa["question"]).lower()
            if gt in ans or animal.lower() in ans or "horse" in ans or "animal" in ans or "cow" in ans or "buffalo" in ans or "ox" in ans:
                correct += 1
            total += 1
        if (i + 1) % CACHE_EVERY == 0:
            gc.collect(); torch.cuda.empty_cache()
    return (correct / total * 100) if total else 0.0


# ─── LL: Language Leakage ─────────────────────────────────────────────────────
def eval_language_leakage(probes):
    """LL: % of text-only probes where the model mentions zebra stripes. Lower = better."""
    leaked = 0
    for probe in tqdm(probes, desc="LL (Language Leakage)", leave=False):
        ans = ask_vlm(None, probe["question"], max_new_tokens=80)
        # Check if any leak keyword appears in the answer
        if any(kw in ans.lower() for kw in probe["leak_keywords"]):
            leaked += 1
    return (leaked / len(probes) * 100) if probes else 0.0


# ─── AR: Adversarial Robustness ───────────────────────────────────────────────
def eval_adversarial(adv_images):
    """AR: % of blurred/greyscale zebra images still identified as zebra. Lower = better."""
    correct = total = 0
    for i, pair in enumerate(tqdm(adv_images, desc="AR (Adversarial)", leave=False)):
        for variant in ["blur", "grey"]:
            ans = ask_vlm(pair[variant], "What animal is shown in this image?")
            if "zebra" in ans.lower():
                correct += 1
            total += 1
        if (i + 1) % CACHE_EVERY == 0:
            gc.collect(); torch.cuda.empty_cache()
    return (correct / total * 100) if total else 0.0


def run_full_eval(label):
    """Run all four metrics and return a results dict."""
    print(f"\n{'='*50}")
    print(f"  Evaluating: {label}")
    print(f"{'='*50}")
    fa = eval_forget(forget_images, forget_qa)
    print(f"  FA  = {fa:.2f}%")
    ra = eval_retain(retain_images, retain_labels, retain_qa)
    print(f"  RA  = {ra:.2f}%")
    ll = eval_language_leakage(LANGUAGE_LEAKAGE_PROBES)
    print(f"  LL  = {ll:.2f}%")
    ar = eval_adversarial(adversarial_images)
    print(f"  AR  = {ar:.2f}%")
    gc.collect(); torch.cuda.empty_cache()
    return {"label": label, "FA": round(fa,2), "RA": round(ra,2), "LL": round(ll,2), "AR": round(ar,2)}


print("✅ Evaluation functions defined")

✅ Evaluation functions defined


## 10. Baseline (No Steering)

In [16]:
results_baseline = run_full_eval("Baseline (No Steering)")


  Evaluating: Baseline (No Steering)


FA (Forget):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  FA  = 100.00%


RA (Retain):   0%|          | 0/20 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


  LL  = 100.00%


AR (Adversarial):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  AR  = 87.50%


## 11. Case 1: Vision-Only SAE Steering

Hooks into **CLIP Layer 18** inside LLaVA and applies additive steering to suppress zebra-selective visual features.  
The LM decoder is untouched — if stripes information leaks through the LM's prior knowledge, LL may stay high.

In [17]:
# Activate vision steering hook ONLY
h_vis = activate_vision_steering()
print(f"🟢 Vision steering ACTIVE (alpha={VIS_ALPHA}, {VIS_N_FEATURES} features @ CLIP layer {VIS_SAE_LAYER})")

results_vision_only = run_full_eval("Vision-Only SAE")

h_vis.remove()
print("🔴 Vision steering REMOVED")
gc.collect(); torch.cuda.empty_cache()

🟢 Vision steering ACTIVE (alpha=-3.0, 50 features @ CLIP layer 18)

  Evaluating: Vision-Only SAE


FA (Forget):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  FA  = 33.33%


RA (Retain):   0%|          | 0/20 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


  LL  = 100.00%


AR (Adversarial):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  AR  = 0.00%
🔴 Vision steering REMOVED


## 12. Case 2: Language-Only SAE Steering

Hooks into **LLaMA Layer 24** and applies negative clamping to suppress zebra-specific language features.  
The CLIP vision encoder is untouched — the model "sees" the zebra but can't articulate it.

In [18]:
# Activate language steering hook ONLY
h_lm = activate_language_steering()
print(f"🟢 Language steering ACTIVE (clamp={LM_CLAMP_VALUE}, {LM_N_FEATURES} features @ LM layer 24)")

results_language_only = run_full_eval("Language-Only SAE")

h_lm.remove()
print("🔴 Language steering REMOVED")
gc.collect(); torch.cuda.empty_cache()

🟢 Language steering ACTIVE (clamp=-4.4, 20 features @ LM layer 24)

  Evaluating: Language-Only SAE


FA (Forget):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  FA  = 0.00%


RA (Retain):   0%|          | 0/20 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


  LL  = 40.00%


AR (Adversarial):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  AR  = 0.00%
🔴 Language steering REMOVED


## 13. Case 3: Both SAEs (Vision + Language)

In [19]:
# Activate BOTH hooks simultaneously
h_vis = activate_vision_steering()
h_lm  = activate_language_steering()
print(f"🟢 BOTH steering hooks ACTIVE")
print(f"   Vision:   alpha={VIS_ALPHA}, {VIS_N_FEATURES} features @ CLIP layer {VIS_SAE_LAYER}")
print(f"   Language: clamp={LM_CLAMP_VALUE}, {LM_N_FEATURES} features @ LM layer 24")

results_both = run_full_eval("Vision + Language SAE")

h_vis.remove()
h_lm.remove()
print("🔴 Both steering hooks REMOVED")
gc.collect(); torch.cuda.empty_cache()

🟢 BOTH steering hooks ACTIVE
   Vision:   alpha=-3.0, 50 features @ CLIP layer 18
   Language: clamp=-4.4, 20 features @ LM layer 24

  Evaluating: Vision + Language SAE


FA (Forget):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  FA  = 0.00%


RA (Retain):   0%|          | 0/20 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


  LL  = 40.00%


AR (Adversarial):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

  AR  = 0.00%
🔴 Both steering hooks REMOVED


## 14. Results Summary

In [20]:
all_results = [results_baseline, results_vision_only, results_language_only, results_both]

print("\n" + "=" * 85)
print("  ZEBRA UNLEARNING RESULTS — SAE Steering on LLaVA-NeXT")
print("=" * 85)
print(f"  {'Method':<30} {'FA ↓':>8} {'RA ↑':>8} {'LL ↓':>8} {'AR ↓':>8}")
print(f"  {'(lower FA/LL/AR = better unlearning)':<30} {'(forget)':>8} {'(retain)':>8} {'(lang.)':>8} {'(adv.)':>8}")
print("  " + "-" * 75)
for r in all_results:
    print(f"  {r['label']:<30} {r['FA']:>7.2f}% {r['RA']:>7.2f}% {r['LL']:>7.2f}% {r['AR']:>7.2f}%")
print("=" * 85)

# Delta vs baseline
b = results_baseline
print("\n  Delta vs Baseline:")
print(f"  {'Method':<30} {'ΔFA':>8} {'ΔRA':>8} {'ΔLL':>8} {'ΔAR':>8}")
print("  " + "-" * 60)
for r in [results_vision_only, results_language_only, results_both]:
    dFA = r['FA'] - b['FA']; dRA = r['RA'] - b['RA']
    dLL = r['LL'] - b['LL']; dAR = r['AR'] - b['AR']
    print(f"  {r['label']:<30} {dFA:>+7.2f}% {dRA:>+7.2f}% {dLL:>+7.2f}% {dAR:>+7.2f}%")
print("=" * 85)

print("\nInterpretation Guide:")
print("  FA ↓  = model less likely to identify zebra in forget images (unlearning success)")
print("  RA ↑  = model still correctly identifies horses (no collateral damage)")
print("  LL ↓  = model less likely to describe zebra stripes in text-only queries")
print("  AR ↓  = model fails to identify zebra even in blurred/greyscale images")

# Save to JSON
with open("/kaggle/working/sae_unlearning_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\n📄 Results saved to /kaggle/working/sae_unlearning_results.json")


  ZEBRA UNLEARNING RESULTS — SAE Steering on LLaVA-NeXT
  Method                             FA ↓     RA ↑     LL ↓     AR ↓
  (lower FA/LL/AR = better unlearning) (forget) (retain)  (lang.)   (adv.)
  ---------------------------------------------------------------------------
  Baseline (No Steering)          100.00%   98.33%  100.00%   87.50%
  Vision-Only SAE                  33.33%   66.67%  100.00%    0.00%
  Language-Only SAE                 0.00%   88.33%   40.00%    0.00%
  Vision + Language SAE             0.00%   51.67%   40.00%    0.00%

  Delta vs Baseline:
  Method                              ΔFA      ΔRA      ΔLL      ΔAR
  ------------------------------------------------------------
  Vision-Only SAE                 -66.67%  -31.66%   +0.00%  -87.50%
  Language-Only SAE              -100.00%  -10.00%  -60.00%  -87.50%
  Vision + Language SAE          -100.00%  -46.66%  -60.00%  -87.50%

Interpretation Guide:
  FA ↓  = model less likely to identify zebra in forget image

## 15. Sanity Check: Qualitative Examples

In [21]:
test_img = forget_images[0]
test_q   = "What animal is shown in this image? Describe its appearance."

print("=" * 60)
print("QUALITATIVE CHECK — Forget image (zebra)")
print("=" * 60)

# Baseline
print(f"\n[BASELINE]\n  {ask_vlm(test_img, test_q, 80)}")

# Vision only
h = activate_vision_steering()
print(f"\n[VISION-ONLY STEERED]\n  {ask_vlm(test_img, test_q, 80)}")
h.remove()

# Language only
h = activate_language_steering()
print(f"\n[LANGUAGE-ONLY STEERED]\n  {ask_vlm(test_img, test_q, 80)}")
h.remove()

# Both
hv = activate_vision_steering(); hl = activate_language_steering()
print(f"\n[BOTH STEERED]\n  {ask_vlm(test_img, test_q, 80)}")
hv.remove(); hl.remove()

# Retain sanity
print("\n" + "-" * 60)
print("RETAIN IMAGE (horse) — Both hooks active:")
hv = activate_vision_steering(); hl = activate_language_steering()
print(f"  {ask_vlm(retain_images[0], 'What animal is this?', 60)}")
hv.remove(); hl.remove()

# Language leakage sanity
print("\n" + "-" * 60)
print("LANGUAGE LEAKAGE — Both hooks active:")
hv = activate_vision_steering(); hl = activate_language_steering()
print(f"  Q: Describe the coat pattern of a zebra.")
print(f"  A: {ask_vlm(None, 'Describe the typical coat pattern of a zebra in detail.', 80)}")
hv.remove(); hl.remove()

gc.collect(); torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


QUALITATIVE CHECK — Forget image (zebra)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



[BASELINE]
  The image shows a group of zebras and a yak. The zebras have distinctive black and white striped coats, which are characteristic of their species. They have a mane that runs along the back of their necks, and their tails are also striped. The yak, on the other hand, is a large, horned animal with a thick, shaggy coat that is typically brown or


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...


[LANGUAGE-ONLY STEERED]
  The image shows a group of animals, but the one that is most visible and the most common in the image is the z
Be the first to be the green green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green and the green


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



[BOTH STEERED]
  There is no animal in the image. The image is a view of a green park, a blue sky, a blue and green water, a blue and green ground, a blue and green tree, a blue and green tree, a blue and green tree, a blue and green tree, a blue and green tree, a blue and green tree, a blue and green tree, a blue and green

------------------------------------------------------------
RETAIN IMAGE (horse) — Both hooks active:


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


  The image you've provided is a high resolution, color, and the best part of the image is the high resolution, color, and the best part of the image is the high resolution, color, and the best part of the image is the high resolution, color, and the best part of the

------------------------------------------------------------
LANGUAGE LEAKAGE — Both hooks active:
  Q: Describe the coat pattern of a zebra.
  A: The typical coat of a zear is a black and white pattern. The black and white color of the coat is a result of the presence of a pigment called melanin. The black and white color of the coat is a result of the presence of a pigment called melanin. The black and white color of the coat is a result of the presence of a pigment called melanin. The black
